In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

In [2]:
# =========================================================
# 1) PATH SETUP: Kaggle + Local uyumlu
# =========================================================
KAGGLE_BASE = "/kaggle/input/competitions/playground-series-s6e3/"
LOCAL_BASE = "../data/raw/"   # localde train.csv, test.csv, sample_submission.csv aynı klasördeyse

if os.path.exists(os.path.join(KAGGLE_BASE, "train.csv")):
    BASE_PATH = KAGGLE_BASE
    ENVIRONMENT = "Kaggle"
elif os.path.exists(os.path.join(LOCAL_BASE, "train.csv")):
    BASE_PATH = LOCAL_BASE
    ENVIRONMENT = "Local"
else:
    raise FileNotFoundError(
        "train.csv bulunamadı. Kaggle dizinini veya local klasörü kontrol et."
    )

print(f"Çalışma ortamı: {ENVIRONMENT}")
print(f"Kullanılan yol : {BASE_PATH}")

train_path = os.path.join(BASE_PATH, "train.csv")
test_path = os.path.join(BASE_PATH, "test.csv")
sub_path = os.path.join(BASE_PATH, "sample_submission.csv")


Çalışma ortamı: Local
Kullanılan yol : ../data/raw/


In [3]:
# =========================================================
# 2) DATA LOAD
# =========================================================
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Sub shape  : {sub.shape}")

Train shape: (594194, 21)
Test shape : (254655, 20)
Sub shape  : (254655, 2)


In [4]:

# =========================================================
# 3) BASIC SETUP
# =========================================================
TARGET = "Churn"
ID_COL = "id"
RANDOM_STATE = 42
N_SPLITS = 3

# Target mapping
# Churn sütunu No/Yes geliyor
train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1})

if train[TARGET].isnull().sum() > 0:
    bad_values = train.loc[train[TARGET].isnull(), TARGET].unique()
    raise ValueError(f"Churn sütununda beklenmeyen değerler var: {bad_values}")

X = train.drop(columns=[ID_COL, TARGET]).copy()
y = train[TARGET].astype(int).copy()
X_test = test.drop(columns=[ID_COL]).copy()

# Kategorik sütunları otomatik bul
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
cat_features_idx = [X.columns.get_loc(col) for col in cat_cols]

print(f"\nFeature sayısı    : {X.shape[1]}")
print(f"Kategorik sütun   : {len(cat_cols)}")
print(f"Churn oranı       : {y.mean():.2%}")
print(f"Kategorik kolonlar: {cat_cols}")



Feature sayısı    : 19
Kategorik sütun   : 15
Churn oranı       : 22.52%
Kategorik kolonlar: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### yarışta 
3 günlük bir yarışta hedef maksimum skor değil, hızlı iterasyon olmalı

iterations = 300
n_splits = 3 

In [5]:
# =========================================================
# 4) CROSS VALIDATION
# =========================================================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    oof_preds[valid_idx] = valid_pred
    test_preds += test_pred / N_SPLITS

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(fold_auc)

    print(f"Fold {fold} AUC: {fold_auc:.5f}")


0:	test: 0.8731376	best: 0.8731376 (0)	total: 1.2s	remaining: 5m 59s
200:	test: 0.9126613	best: 0.9126613 (200)	total: 5m 49s	remaining: 2m 51s
299:	test: 0.9134543	best: 0.9134543 (299)	total: 9m 23s	remaining: 0us

bestTest = 0.9134543056
bestIteration = 299

Fold 1 AUC: 0.91345
0:	test: 0.8735175	best: 0.8735175 (0)	total: 1.96s	remaining: 9m 45s
200:	test: 0.9137226	best: 0.9137226 (200)	total: 5m 11s	remaining: 2m 33s
299:	test: 0.9144252	best: 0.9144252 (299)	total: 6m 41s	remaining: 0us

bestTest = 0.9144252238
bestIteration = 299

Fold 2 AUC: 0.91443
0:	test: 0.8752560	best: 0.8752560 (0)	total: 845ms	remaining: 4m 12s
200:	test: 0.9120978	best: 0.9120978 (200)	total: 2m 49s	remaining: 1m 23s
299:	test: 0.9128579	best: 0.9128579 (299)	total: 3m 37s	remaining: 0us

bestTest = 0.9128578762
bestIteration = 299

Fold 3 AUC: 0.91286


In [6]:
# =========================================================
# 5) CV RESULT
# =========================================================
cv_auc = roc_auc_score(y, oof_preds)

print("\n" + "=" * 55)
print("Baseline CV Results")
print("=" * 55)
print(f"Fold scores : {[round(s, 5) for s in fold_scores]}")
print(f"Mean AUC    : {np.mean(fold_scores):.5f}")
print(f"Std AUC     : {np.std(fold_scores):.5f}")
print(f"OOF AUC     : {cv_auc:.5f}")


Baseline CV Results
Fold scores : [np.float64(0.91345), np.float64(0.91443), np.float64(0.91286)]
Mean AUC    : 0.91358
Std AUC     : 0.00065
OOF AUC     : 0.91357


In [7]:
# =========================================================
# 6) SUBMISSION
# =========================================================
sub[TARGET] = test_preds

# submission klasörü
OUTPUT_DIR = "../outputs/submissions"

# klasör yoksa oluştur
os.makedirs(OUTPUT_DIR, exist_ok=True)

submission_name = os.path.join(OUTPUT_DIR, "submission_baseline_catboost.csv")

sub.to_csv(submission_name, index=False)

print(f"\nSubmission dosyası oluşturuldu: {submission_name}")
print(sub.head())


Submission dosyası oluşturuldu: ../outputs/submissions\submission_baseline_catboost.csv
       id     Churn
0  594194  0.070582
1  594195  0.001880
2  594196  0.090184
3  594197  0.002644
4  594198  0.486197
